# S47_01 — Why RAG?

**Retrieval-Augmented Generation (RAG)** combines information retrieval with LLM generation. Instead of relying on the LLM's parametric memory, you fetch relevant documents at query time and include them in the prompt.

## The problem RAG solves

LLMs have three fundamental limitations:

1. **Knowledge cutoff** — training data has a fixed date; the model doesn't know about events after it
2. **Hallucination** — models confabulate facts that sound plausible but aren't grounded in sources
3. **Context window constraints** — you can't fit an entire document corpus into every prompt

RAG addresses all three: it fetches current, relevant documents, gives the model verifiable sources to cite, and uses retrieval to fit only the relevant slice into the context window.

In [ ]:
# Minimal RAG concept — manual version before using frameworks
import anthropic

client = anthropic.Anthropic()

# Simulated document store
documents = {
    'doc1': 'Python was created by Guido van Rossum and first released in 1991.',
    'doc2': 'PyTorch was released by Meta AI Research in 2016. It uses dynamic computation graphs.',
    'doc3': 'NumPy is a library for numerical computation in Python, providing N-dimensional arrays.',
    'doc4': 'pandas was created by Wes McKinney in 2008 for data manipulation and analysis.',
}

def naive_retrieve(query, docs, top_k=2):
    """Keyword overlap retrieval — naive but illustrative."""
    query_words = set(query.lower().split())
    scores = {}
    for doc_id, text in docs.items():
        doc_words = set(text.lower().split())
        scores[doc_id] = len(query_words & doc_words)
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [(doc_id, docs[doc_id]) for doc_id in ranked[:top_k] if scores[doc_id] > 0]

def rag_answer(query):
    retrieved = naive_retrieve(query, documents)
    context = '\n\n'.join(f'[{doc_id}]: {text}' for doc_id, text in retrieved)
    
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=256,
        system='Answer questions using ONLY the provided context. If the context does not contain the answer, say "I don\'t have information about that."',
        messages=[{
            'role': 'user',
            'content': f'Context:\n{context}\n\nQuestion: {query}'
        }],
    )
    return msg.content[0].text, retrieved

answer, sources = rag_answer('When was PyTorch released?')
print('Answer:', answer)
print('Sources:', [s[0] for s in sources])

## RAG vs. alternatives

| Approach | When to use | Limitation |
|----------|-------------|------------|
| **RAG** | Dynamic, frequently updated knowledge | Retrieval quality bottleneck |
| **Fine-tuning** | Domain style/format, task specialization | Doesn't add new facts reliably |
| **Long context** | All docs fit in window (<200k tokens) | Cost, latency, lost-in-middle problem |
| **Tool use** | Real-time data (weather, stocks) | Requires API access |

**RAG and fine-tuning are complementary**: fine-tune for style/task format, RAG for factual grounding.

In [ ]:
# Demonstrate hallucination without RAG
def raw_llm_answer(query):
    msg = client.messages.create(
        model='claude-haiku-4-5-20251001',
        max_tokens=128,
        messages=[{'role': 'user', 'content': query}],
    )
    return msg.content[0].text

# Ask about something that might produce a hallucination — company-specific policy
query = 'What is the return policy at Acme Corp?'

print('Without RAG:')
print(raw_llm_answer(query))
print()

# Add a policy document to the store
documents['acme_policy'] = 'Acme Corp return policy: All items can be returned within 30 days with receipt. Electronics have a 14-day return window.'

print('With RAG:')
answer, sources = rag_answer(query)
print(answer)
print('Source:', sources[0][0] if sources else 'none')

## The RAG pipeline

```
Documents → [Chunking] → [Embedding] → [Vector Store]
                                              ↑
Query → [Embedding] → [Retrieval] → [Reranking] → [LLM] → Answer
```

- **Chunking** — split documents into pieces that fit in context (see S47_02)
- **Embedding** — convert text to dense vectors (see S47_03)
- **Vector store** — index for fast approximate nearest-neighbor search (see S47_03)
- **Retrieval strategies** — dense, sparse, hybrid, reranking (see S47_04)
- **End-to-end** — putting it all together (see S47_05)

Next: [S47_02_document_loading_chunking.ipynb](./S47_02_document_loading_chunking.ipynb)